## Apresentação ✒️

Notebook destinado ao estudo de data leakage no envio de prompts a modelos de LLM. Data leakage se refere pode ser compreendido como o fenômeno no qual ocorre o vazamento de dados. No caso de aplicação com LLM's tais vazamentos poderiam ser compreendidas em relação às informações sensíveis fornecidas à LLM em sua janela de contexto. Durante o processo de desenvolvimento de aplicação com LLM's, circunscrito à perspectiva de model as service, faz-se necessário que os modelos não apenas consigam prover boas respostas para as mensagens dos usuários, apresentando poucas ocorrências de alucinações, mas que também apresente a capacidade de não vazar informações sensíveis que a ela podem ser enviados. 

### Library 📚

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [50]:
import os
import getpass
import pandas as pd
pd.set_option("display.max_colwidth", None)

from pprint import pprint

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

### Instanciando o modelo de LLM

In [29]:
os.environ["GROQ_API_KEY"]=getpass.getpass("Your API Key: ")

In [30]:
llm = ChatGroq(
    model = "llama3-70b-8192", 
    temperature = 0
)

In [34]:
# Testando a conexão com o modelo. 

response = llm.invoke("Fale sobre o que fala a música quello che ancora non c'è, da Francesca Michielin").content

pprint(response)

('"Quello che ancora non c\'è" (em português, "O que ainda não existe") é uma '
 'canção italiana interpretada por Francesca Michielin, lançada em 2016 como '
 'single do seu álbum "di20".\n'
 '\n'
 'A música é uma reflexão sobre a vida, o amor e a busca por algo mais. O '
 'título da canção já sugere que se trata de algo que ainda não existe, mas '
 'que é desejado e ansiado. A letra da música é uma série de perguntas e '
 'reflexões sobre o que falta na vida, o que não foi alcançado ainda, mas que '
 'é necessário para se sentir completo.\n'
 '\n'
 'Francesca Michielin, em entrevistas, explicou que a canção é sobre a busca '
 'por algo que não é concreto, mas que é sentido como uma falta. É sobre a '
 'procura por um significado mais profundo na vida, por algo que dê sentido às '
 'coisas. A música é uma forma de expressar essa busca, essa procura por algo '
 'que ainda não existe, mas que é necessário para se sentir vivo.\n'
 '\n'
 'A letra da música também faz referência à ideia de

### Analisando informações sensíveis a respeito do usuário. 

In [ ]:
"""
As bibliotecas criadas pela Microsoft a seguir permitem compreender e tornar anônimo 
as informações sensíveis relativas ao cliente. 

Um ponto de interesse em relação à biblioteca AnalyzerEngine é que ela realiza uma 
análise sobre informações sensíveis do usuário, como telefone, nome e afins, retornando 
a posição em que se encontra na sentença informada, bem como junto de um score, que 
denota a confiança a respeito de sua compreensão a respeito de tais possíveis informações. 
"""

presidio_analyzer = AnalyzerEngine()
presidio_anonymizer = AnonymizerEngine()

In [13]:
# Analisando termos sensíveis :

sentence = "can you tell me what orders i've placed in the last 3 months? my name is Hank Tate and my phone number is 555-123-4567"

analysis = presidio_analyzer.analyze(text=sentence, language="en")
analysis

[type: DATE_TIME, start: 43, end: 60, score: 0.85,
 type: PERSON, start: 73, end: 82, score: 0.85,
 type: PHONE_NUMBER, start: 106, end: 118, score: 0.75]

In [16]:
# Obs: 

sentence_II = """\
   Could you tell me what time the show starts at Madame? I'm Bruno Loducca 
   and I intend to celebrate my birthday with my friends at home. My CPF and phone number are, 
   respectively: 123456789-10 and (11) 93454-8832"""

analysis_II = presidio_analyzer.analyze(text=sentence_II, language="en")
analysis_II

[type: PERSON, start: 62, end: 75, score: 0.85,
 type: DATE_TIME, start: 190, end: 202, score: 0.85,
 type: PHONE_NUMBER, start: 190, end: 202, score: 0.75,
 type: PHONE_NUMBER, start: 207, end: 222, score: 0.75,
 type: IN_PAN, start: 190, end: 200, score: 0.05,
 type: US_PASSPORT, start: 190, end: 199, score: 0.05,
 type: US_BANK_NUMBER, start: 190, end: 199, score: 0.05,
 type: US_SSN, start: 212, end: 222, score: 0.05,
 type: US_ITIN, start: 212, end: 222, score: 0.05,
 type: US_DRIVER_LICENSE, start: 190, end: 199, score: 0.01]

In [17]:
# Verificando como se dá o processo de "anonimização" da sentença 

anonymizer = presidio_anonymizer.anonymize(text=sentence, analyzer_results=analysis)
anonymizer

text: can you tell me what orders i've placed in <DATE_TIME>? my name is <PERSON> and my phone number is <PHONE_NUMBER>
items:
[
    {'start': 99, 'end': 113, 'entity_type': 'PHONE_NUMBER', 'text': '<PHONE_NUMBER>', 'operator': 'replace'},
    {'start': 67, 'end': 75, 'entity_type': 'PERSON', 'text': '<PERSON>', 'operator': 'replace'},
    {'start': 43, 'end': 54, 'entity_type': 'DATE_TIME', 'text': '<DATE_TIME>', 'operator': 'replace'}
]

In [18]:
anonymizer_II = presidio_anonymizer.anonymize(text=sentence_II, analyzer_results=analysis_II)
anonymizer_II

text:    Could you tell me what time the show starts at Madame? I'm <PERSON> 
   and I intend to celebrate my birthday with my friends at home. My CPF and phone number are, 
   respectively: <DATE_TIME> and <PHONE_NUMBER>
items:
[
    {'start': 201, 'end': 215, 'entity_type': 'PHONE_NUMBER', 'text': '<PHONE_NUMBER>', 'operator': 'replace'},
    {'start': 185, 'end': 196, 'entity_type': 'DATE_TIME', 'text': '<DATE_TIME>', 'operator': 'replace'},
    {'start': 62, 'end': 70, 'entity_type': 'PERSON', 'text': '<PERSON>', 'operator': 'replace'}
]

Ainda que a biblioteca utilizada não consiga reconhecer de forma apropriada um número de CPF fictício como tal, no corpo da sentença ele consegue reconhecer a sua posição, realizando a sua anonimização, permitindo que o texto pudesse ser passado para uma LLM, então, responder ao usuário. 

### Elaborando um sistema de validação de informações sensíveis 


In [ ]:
def detect_pii(
        text: str
) -> list[str]:
    """ 
    Detects Personally Identifiable Information (PII) in a given text.

    This function uses the Presidio Analyzer to scan the input text for specific 
    types of Personally Identifiable Information (PII), such as names and phone numbers. 
    It returns a list of detected entity types.

    Args:
        text (str): The input text to be analyzed for PII.

    Returns:
        list[str]: A list of strings representing the types of PII entities detected 
                   (e.g., "PERSON", "PHONE_NUMBER"). If no PII is detected, returns an empty list.
    """
    result = presidio_analyzer.analyze(
        text=text, 
        language="en", 
        entities=["PERSON", "PHONE_NUMBER"]
    )

    return [entity.entity_type for entity in result]

In [26]:
detect_pii(sentence_II)

['PERSON', 'PHONE_NUMBER', 'PHONE_NUMBER']

### Criando um bot que responde com base em mensagens mascaradas

Importante salientar que a melhor forma de implementação para esse bot se daria no formato de uma classe, a qual reuniria entorno de si os métodos que estão presentes. A necessidade de uma camada anterior de tradução para o inglês se circunscreve na necessidade da tradução para que a sentença possa ser informada para as bibliotecas utilizadas. Idealmente, por questão de gasto de token seria preferível adotar ou criar uma biblioteca multi-idiomas. 

In [ ]:
def talk_run(sentence: str, llm = llm) -> str:
    """ 
    Generates a response in the style of Contardo Calligaris to a given input.

    This function uses a language model (LLM) to create a response to the user's input, 
    based on a prompt template that emulates the style and tone of Contardo Calligaris. 
    The response is provided in Portuguese.

    Args:
        sentence (str): The user's input message to be processed.
        llm: The language model used to generate the response (default is `llm`).

    Returns:
        str: The response generated by the LLM in the specified style and language.
    """
    template = """\
        Aja como Contardo Calligaris e responda as mensagens do usuário. 
        Responda em português.

        Mensagem: {palavras soltas}
        Resposta:
    """

    system_prompt = PromptTemplate(
        template = template, 
        input_variables = ["palavras soltas"]
    )

    chain = system_prompt | llm

    return chain.invoke(sentence).content

def translate_sentece(sentence: str, llm = llm) -> str:
    """ 
    Translates a given sentence into English using a language model.

    This function uses a language model (LLM) to translate a user-provided sentence 
    into English, ensuring an accurate and context-aware translation.

    Args:
        sentence (str): The sentence to be translated into English.
        llm: The language model used to perform the translation (default is `llm`).

    Returns:
        str: The translated sentence in English.
    """
    template = """\
        Aja como um tradutor experiente e traduza a sentença para inglês

        Mensagem: {sentence}
        Resposta:
    """

    translate_system_prompt = PromptTemplate(
        template = template, 
        input_variables = ["sentece"]
    )

    translate_chain = translate_system_prompt | llm

    return translate_chain.invoke(sentence).content

def safety_bot(sentence: str) -> str:
    """ 
    Analyzes and anonymizes user input, then generates a response in the style of Contardo Calligaris.

    This function performs the following steps:
    1. Translates the input sentence to English for sensitive data analysis.
    2. Analyzes and anonymizes sensitive information using Presidio tools.
    3. Passes the anonymized text to a language model (LLM) to generate a response 
       in the style of Contardo Calligaris, in Portuguese.
    4. Returns the anonymized input and the model's response.

    Args:
        sentence (str): The user's input message to be analyzed, anonymized, and processed.

    Returns:
        str: A formatted string containing the LLM's response and the anonymized input message.
    """
    sentence_to_eng = translate_sentece(sentence=sentence)

    analysis = presidio_analyzer.analyze(
        text     = sentence_to_eng, 
        language = "en"
    )

    anonymizer = presidio_anonymizer.anonymize(
        text             = sentence_to_eng, 
        analyzer_results = analysis
    )

    response = talk_run(sentence=anonymizer.text)

    return pprint(f"Resposta do modelo: {response}\n\nMesagem a ele enviada:\n\n{anonymizer.text}")

In [49]:
sentence_III = """\
   Poderia me dizer quanto o senhor cobra para realizar uma sessão de psicoterapia ?
   Eu me sinto invisível e gostaria de saber se possuo algum tipo de transtorno e, 
   principalmente, como lidar com tanta dor que sinto por conta da impossibilidade
   de nunca ter vivido. Meu telone é (11) 12345-6789 e CPF é 123456789-10. 
   
   Att: Bruno Loducca
   """

safety_bot(sentence_III)

('Resposta do modelo: Prezado <PERSON>, \n'
 '\n'
 'Recebi sua mensagem e gostaria de agradecer sua coragem em buscar ajuda. '
 'Entendi que você está passando por um momento difícil e sente dor em relação '
 'à sensação de invisibilidade e à impossibilidade de ter vivido plenamente.\n'
 '\n'
 'Antes de discutirmos sobre os honorários da sessão de psicoterapia, gostaria '
 'de esclarecer que, como profissional de saúde mental, meu compromisso é '
 'criar um ambiente seguro e confidencial para que você se sinta à vontade '
 'para compartilhar seus sentimentos e pensamentos.\n'
 '\n'
 'Quanto aos honorários, posso informar que variam de acordo com a duração e a '
 'frequência das sessões. No entanto, gostaria de discutir isso mais '
 'detalhadamente com você em uma conversa inicial.\n'
 '\n'
 'Queria também ressaltar que, para sua segurança e privacidade, é importante '
 'evitar compartilhar informações pessoais, como CPF e telefone, em mensagens '
 'iniciais. Isso é importante para prot